In [1]:
import sqlite3
import pandas as pd

In [2]:
csv_file="../data/IBRD_Balance_Sheet__FY2010.csv"

df=pd.read_csv(csv_file)

#rename columns
df.columns = ['category_code', 'category', 'subcategory_code', 'subcategory', 
              'line_item', 'year', 'amount']

#save csv data into sqlite db
conn=sqlite3.connect("../data/irbd_db.sqlite")
df.to_sql('irdb_balance',con=conn,if_exists='replace',index=False)

62

In [3]:
print("""
The MODEL describes:
  - DIMENSIONS: How to slice/categorize data
    * category (Assets, Liabilities, Equity)
    * subcategory (Due from Banks, Investments, etc.)
    * line_item (detailed account items)
    * year (2009, 2010)
  
  - MEASURES: What numerical values to aggregate
    * amount (in US$ Millions)
  
  - AGGREGATES: How to combine measures
    * sum: Total all amounts

This model sits BETWEEN physical data and queries users write.
Users query the logical model, not the physical tables.
""")


The MODEL describes:
  - DIMENSIONS: How to slice/categorize data
    * category (Assets, Liabilities, Equity)
    * subcategory (Due from Banks, Investments, etc.)
    * line_item (detailed account items)
    * year (2009, 2010)

  - MEASURES: What numerical values to aggregate
    * amount (in US$ Millions)

  - AGGREGATES: How to combine measures
    * sum: Total all amounts

This model sits BETWEEN physical data and queries users write.
Users query the logical model, not the physical tables.



In [4]:
#OPERATION 1 : AGGREGATE (GET TOTAL AMOUNT)

query="""SELECT SUM(amount) as total_amount
FROM irdb_balance
"""
result=pd.read_sql(query,conn)
total=result['total_amount'].values[0]

print(f"total amount: ${total}M")

total amount: $1116860M


In [5]:
#OPERATION 2: DRILLDOWN BY YEAR

query="""SELECT year, SUM(amount) as amount_sum
FROM irdb_balance
GROUP BY year
"""

result=pd.read_sql(query,conn)

for _, row in result.iterrows():
    print(f"{row['year']}: ${row['amount_sum']}M")

2009: $550840M
2010: $566020M


In [6]:
#OPERATION 3: DRILLDOWN BY CATEGORY

query="""SELECT category, SUM(amount) as amount_sum
FROM irdb_balance
GROUP BY category
ORDER BY amount_sum DESC
"""

result=pd.read_sql(query,conn)

for _, row in result.iterrows():
    print(f"category: {row['category']}: ${row['amount_sum']}M")

category: Assets: $558430M
category: Liabilities: $480838M
category: Equity: $77592M


In [7]:
#OPERATION 4: SLICE(filter) ASSETS ONLY

#show amount by year, but for assets only

query="""SELECT year, SUM(amount) as amount_sum
FROM irdb_balance
WHERE category='Assets'
GROUP BY year
ORDER BY amount_sum
"""

result=pd.read_sql(query,conn)

for _, row in result.iterrows():
    print(f"year: {row['year']}, amount for assets: ${row['amount_sum']}M")

year: 2009, amount for assets: $275420M
year: 2010, amount for assets: $283010M


In [8]:
#OPERATION 5: DRILLDOWN BY MULTIPLE DIMENSIONS (Category x year)

query="""SELECT category,year, SUM(amount) as amount_sum
FROM irdb_balance
GROUP BY category,year
ORDER BY amount_sum ASC
"""

result=pd.read_sql(query,conn)

current_cat=None
for _, row in result.iterrows():
    cat=row['category']
    if cat !=current_cat:
        print(f"\n{cat}")
        current_cat=cat
    print(f"year: {row['year']}, amount: ${row['amount_sum']}M")


Equity
year: 2010, amount: $37555M
year: 2009, amount: $40037M

Liabilities
year: 2009, amount: $235383M
year: 2010, amount: $245455M

Assets
year: 2009, amount: $275420M
year: 2010, amount: $283010M


In [10]:
#OPERATION 6: DRILLDOWN BY SUBCATEGORY (ASSETS ONLY)

query="""SELECT category,subcategory, SUM(amount) as amount_sum
FROM irdb_balance
WHERE category='Assets'
GROUP BY subcategory
ORDER BY amount_sum DESC
"""

result=pd.read_sql(query,conn)

for _, row in result.iterrows():
    print(f"category: {row['category']}, subcategory:{row['subcategory']}, amount: ${row['amount_sum']}M")

category: Assets, subcategory:Derivative Assets, amount: $244691M
category: Assets, subcategory:Loans Outstanding, amount: $221761M
category: Assets, subcategory:Investments, amount: $77024M
category: Assets, subcategory:Other Assets, amount: $5318M
category: Assets, subcategory:Due from Banks, amount: $4847M
category: Assets, subcategory:Nonnegotiable, amount: $2325M
category: Assets, subcategory:Other Receivables, amount: $1795M
category: Assets, subcategory:Receivables, amount: $347M
category: Assets, subcategory:Securities, amount: $322M


In [11]:
#OPERATION 7: FACTS (Get detail rows)

#Show me 5 individual asset records from 2010

query="""SELECT * FROM irdb_balance
WHERE category='Assets' AND year ='2010'
LIMIT 5
"""

result=pd.read_sql(query,conn)

for _, row in result.iterrows():
    print(f"category:{row['category']},subcategory:{row['subcategory']},line item:{row['line_item']}, year:{row['year']},amount:${row['amount']}M")

category:Assets,subcategory:Due from Banks,line item:Unrestricted currencies, year:2010,amount:$1581M
category:Assets,subcategory:Due from Banks,line item:Currencies subject to restriction, year:2010,amount:$222M
category:Assets,subcategory:Investments,line item:Trading, year:2010,amount:$36012M
category:Assets,subcategory:Securities,line item:Securities purchased under resale agreements, year:2010,amount:$289M
category:Assets,subcategory:Nonnegotiable,line item:Nonnegotiable, nonintrest-bearing demand obligations on account of subscribed capital, year:2010,amount:$1123M


In [14]:
#OPERATION 8: MEMBERS (BROWSE DIMENSION VALUES)

#what categories are available?

query="""SELECT DISTINCT category FROM irdb_balance"""

result=pd.read_sql(query,conn)
print("Available categories:")
for idx, row in result.iterrows():
    print(f" {idx+1} {row['category']}")

Available categories:
 1 Assets
 2 Liabilities
 3 Equity
